In [7]:
import sys
from pathlib import Path
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

# Caminho real do notebook 
NOTEBOOK_DIR = Path.cwd() 

# Sobe até a raiz do projeto (shopnow/) 
ROOT = NOTEBOOK_DIR.parents[2] 

# Adiciona ao sys.path 
if str(ROOT) not in sys.path: sys.path.append(str(ROOT)) 

# Agora sim, pode importar 
from pipeline.utils import find_repo_root, get_raw_dir

# 1. Configuração do ambiente

RAWDIR = get_raw_dir()

In [8]:
# 2. Configuração do Faker, Seed e geração dos dados

# Inicializa o Faker e a seed para reprodutibilidade
fake = Faker("pt_BR")
random.seed(42)

def gerar_atendimentos(qtd: int = 3000) -> list[dict]:
    """Gera dados de atendimento propositalmente inconsistentes para etapa de ETL."""

    canais = ["Site", "site", "SITE", "App", "app", "Aplicativo", 
               "Telefone", "telefone", "WhatsApp", "whatsapp", 
               "Chat Online", "chat online"]

    motivos = ["Entrega", "entrega", "Pagamento", "pagamento", 
               "Produto com defeito", "produto com defeito", 
               "Troca", "troca", "Cancelamento"]

    submotivos = ["Atraso", "atraso", "Erro na cobrança", 
                  "erro cobrança", "Produto errado", 
                  "Arrependimento", "arrependimento", 
                  "Estorno", "estorno"]

    prioridades = ["Baixa", "baixa", "Média", "media", 
                   "Alta", "alta", "Crítica", "critica"]

    areas = ["Logística", "logistica", "Financeiro", 
             "financeiro", "Comercial", "Tecnologia"]

    complexidades = ["Baixa", "baixa", "Média", "Alta", "alta"]

    atendimentos = []

    for i in range(qtd):

        data_abertura = fake.date_time_between(start_date="-2y", end_date="now")

        # 15% chance de erro temporal
        erro_temporal = random.choice([True] + [False]*5)

        data_primeira_resposta = data_abertura + timedelta(
            hours=random.randint(-5 if erro_temporal else 1, 12)
        )

        fechado = random.choice([True, True, True, False])

        if fechado:
            data_fechamento = data_abertura + timedelta(
                hours=random.randint(-10 if erro_temporal else 2, 120)
            )
        else:
            data_fechamento = random.choice([
                None,
                "",  # erro proposital
            ])

        quantidade_reaberturas = random.choice([0, 0, 1, 2, 3, -1])  # -1 erro

        tempo_agente_minutos = random.choice([
            random.randint(5, 120),
            -10,  # erro
        ])

        custo_hora_agente = round(random.uniform(20, 100), 2)

        # 20% chance de custo inconsistente
        if random.random() < 0.2:
            custo_atendimento = round(random.uniform(5, 500), 2)
        else:
            custo_atendimento = round((tempo_agente_minutos / 60) * custo_hora_agente, 2)

        valor_compensacao = random.choice([0, 0, 50, 100, -20])  # negativo proposital
        valor_estornado = random.choice([0, 0, 200, 500, -100])  # negativo proposital

        nota_satisfacao = random.choice([
            random.randint(1, 5),
            0,   # inválido
            6    # inválido
        ])

        nps_classificacao = random.choice([
            "Promotor", "promotor", 
            "Neutro", "neutro", 
            "Detrator", "detrator",
            "N/A"
        ])

        houve_reclamacao_publica = random.choice([
            "Sim", "sim", "SIM",
            "Não", "nao", "NAO"
        ])

        cliente_churnou = random.choice([
            "Sim", "Não", "sim", "nao"
        ])

        recompra_30 = random.choice([
            "Sim", "Não", "sim", "nao"
        ])

        atendimentos.append({

            # 🔑 Identificadores (com possível duplicidade)
            "id_atendimento": random.choice([i + 1, random.randint(1, qtd//2)]),
            "id_pedido": random.choice([
                f"PED-{random.randint(10000,99999)}",
                f"ped-{random.randint(10000,99999)}"
            ]),
            "id_cliente": random.choice([
                random.randint(1, 1500),
                None  # cliente inexistente
            ]),
            "id_agente": random.randint(1, 50),

            # 📌 Classificação bagunçada
            "canal": random.choice(canais),
            "motivo_principal": random.choice(motivos),
            "submotivo": random.choice(submotivos),
            "area_responsavel": random.choice(areas),
            "prioridade": random.choice(prioridades),
            "complexidade": random.choice(complexidades),

            # 📅 Datas inconsistentes
            "data_abertura": data_abertura,
            "data_primeira_resposta": data_primeira_resposta,
            "data_fechamento": data_fechamento,
            "quantidade_reaberturas": quantidade_reaberturas,

            # ⏱ Métricas possivelmente erradas
            "tempo_agente_minutos": tempo_agente_minutos,

            # 💰 Custos inconsistentes
            "custo_hora_agente": custo_hora_agente,
            "custo_atendimento": custo_atendimento,
            "valor_compensacao": valor_compensacao,
            "valor_estornado": valor_estornado,

            # 😊 Qualidade inconsistente
            "nota_satisfacao": nota_satisfacao,
            "nps_classificacao": nps_classificacao,

            # 🎯 Estratégico bagunçado
            "houve_reclamacao_publica": houve_reclamacao_publica,
            "cliente_churnou_apos_atendimento": cliente_churnou,
            "houve_recompra_30_dias": recompra_30
        })

    return atendimentos


In [9]:
# 3. Gera o DataFrame 
df_atendimento = pd.DataFrame(gerar_atendimentos(10000))
#df_atendimento

In [10]:
# 4. Salva o DataFrame como CSV no diretório de dados brutos
df_atendimento.to_csv(
    RAWDIR / "atendimento_raw.csv",
    index=False,
    encoding="utf-8"
)